# 종합문제풀이 (2) — 90분 모의고사

**규칙**: 90분 / 3문제 / 문제당 약 30분. 풀이 노트북(`session*.ipynb`)을 **보지 않고** 풉니다. 검색·문서 참고는 시험과 같게 허용.
데이터는 `session12`에 있어야 합니다. 이 노트북은 `practice/mock_exam/`에서 열면 `../data/...` 경로가 맞습니다.

**전략**: ① 지문 정독 → ② 가장 단순한 방식으로 결과 파일부터 저장 → ③ 시간이 남으면 개선 → ④ 맨 끝에 `Restart & Run All`.

In [ ]:
import time, os
START = time.time()
def elapsed(): print(f"경과 {(time.time()-START)/60:.1f}분 / 권장 문제당 30분")
elapsed()

In [ ]:
DATA_DIR = "../data/session12"   # ← 공식 실습파일 폴더를 이 경로에 복사해 두세요
OUT_DIR = "../outputs/session12"
os.makedirs(OUT_DIR, exist_ok=True)
print(os.listdir(DATA_DIR))

In [ ]:
import os, pandas as pd
def check(path, cols=None, rows=None):
    """제출 파일 자가 점검 (시험의 '데이터 검증' 버튼과 비슷한 역할)"""
    if not os.path.exists(path): print(f"❌ {path} 없음"); return
    d = pd.read_csv(path)
    ok = True
    if cols and not set(cols) <= set(d.columns): print(f"❌ 컬럼 부족: 필요 {cols}, 현재 {list(d.columns)}"); ok = False
    if d.isna().any().any(): print("❌ 결측 있음"); ok = False
    if rows and len(d) != rows: print(f"⚠ 행 수 {len(d)} (예상 {rows})"); ok = False
    if "Unnamed: 0" in d.columns: print("❌ index가 저장됨 → to_csv(index=False)"); ok = False
    print(("✅" if ok else "↑ 확인 필요"), path, d.shape)

## 문제 1. 병사 건강 데이터 기반 이상 감지 (비지도)

**데이터** `health_survey.csv`: `soldier_id`, `bmi`, `sleep_hours`, `exercise_freq`, `smoke`, `alcohol`

**요구사항**
1. 특징 정규화(`StandardScaler`)
2. `IsolationForest`(contamination=0.05)로 이상 탐지, `anomaly` 컬럼(1=이상) 추가
3. 이상 비율 출력
4. `PCA` 2차원 축소 후 이상 여부별 색 산점도 저장(png)
5. 결과 csv 저장

In [ ]:
# ===== 문제 1 풀이 =====
# 결과 파일: psych_anomaly.csv, anomaly_scatter.png


In [ ]:
elapsed()
check(f"{OUT_DIR}/psych_anomaly.csv", cols=["soldier_id","anomaly"])

## 문제 2. 훈련 장면 이미지에서 인원 및 차량 수 탐지 (YOLO)

**데이터** 이미지 폴더 `images/training_scene/` (CAM001~003.png)

**요구사항**
1. YOLO로 각 이미지의 **사람 수**와 **차량 수(car, truck, bus)** 집계
2. csv 저장 (`filename`, `person_count`, `vehicle_count`)
3. 사람 수 vs 차량 수 산점도 저장(png)

> **YOLO 팁**: `torch.hub.load('ultralytics/yolov5','yolov5s')`는 인터넷/깃허브 접근이 필요합니다. 안 되면 `pip install ultralytics` 후 `from ultralytics import YOLO; m = YOLO('yolov8n.pt'); r = m.predict(img_path, device='cpu', verbose=False)[0]; r.boxes.cls.tolist()` 로 클래스 id를 얻습니다(COCO: person=0, car=2, bus=5, truck=7).

In [ ]:
# ===== 문제 2 풀이 =====
# 결과 파일: detection_summary.csv, person_vs_vehicle_scatter.png


In [ ]:
elapsed()
check(f"{OUT_DIR}/detection_summary.csv", cols=["filename","person_count","vehicle_count"], rows=3)

## 문제 3. 체력·심리 데이터로 분류와 회귀 동시 수행

**데이터** `soldier_full_profile.csv`: `soldier_id`, `pushup`, `run_time`, `sleep_quality`, `stress_level`, `status`(fit/unfit), `recovery_days`

**요구사항**
1. 4개 수치 특징을 `StandardScaler`로 정규화, `status`는 `LabelEncoder`(fit=1)
2. **분류**: `LogisticRegression` → 정확도(`accuracy_score`)·혼동행렬 출력
3. **회귀**: `RandomForestRegressor`로 `recovery_days` 예측 → RMSE 출력
4. 두 모델의 결과를 한 산점도에 함께 표시해 `dual_prediction.png`로 저장
5. 전체 예측 결과(`soldier_id`, 실제값, 예측값)를 csv로 저장

(학습/테스트 분리 후 평가할 것)

In [ ]:
# ===== 문제 3 풀이 =====
# 결과 파일: dual_prediction.png, soldier_predictions.csv


In [ ]:
elapsed()
check(f"{OUT_DIR}/soldier_predictions.csv", cols=["soldier_id"])
print(os.path.exists(f"{OUT_DIR}/dual_prediction.png"))

## 채점 & 복기
- 공식 풀이 노트북과 비교: 모델·전처리 선택이 같았나? 평가 방식(분할 후 평가)이 맞았나?
- 막힌 지점을 아래에 적어 두세요.

- 문제 3: 분류와 회귀 각각 `train_test_split` 후 평가. 공식 풀이에 있던 `matplotlob` 오타 같은 실수를 조심하세요.